# Attention-Guided Swin Transformer for Image Compression
## MSc Dissertation 

This notebook covers the training, validation, and ablation study for a novel image compression architecture utilizing Overlapping Patch Tokenizers, Swin Transformers, Spatial/Channel Attention Gates, and an Adaptive Quantizer.

In [ ]:
!pip install compressai timm pytorch-msssim

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from compressai.losses import RateDistortionLoss
from model import AttentionGuidedSwinCompression

# Hyperparameters
EPOCHS = 50
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
LAMBDA_RD = 0.01  # Trade-off between rate and distortion (Quality parameter)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Kaggle Dataset Path - Adjust this to your Kaggle Honeybee UVG dataset path
DATASET_PATH = '/kaggle/input/honeybee-uvg-512x512'

In [ ]:
# Dataset and DataLoader setup
transform = transforms.Compose([
    transforms.RandomCrop(256), # Train on 256x256 crops to save VRAM
    transforms.ToTensor()
])

# Assuming your dataset is structured like `dataset_path/train/images/...`
try:
    train_dataset = ImageFolder(os.path.join(DATASET_PATH, 'train'), transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Dataset not found. Please verify the Kaggle path.")
    # Dummy dataloader for structure
    train_loader = []

In [ ]:
# Model Initialization
model = AttentionGuidedSwinCompression(N=128, M=192).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Rate-Distortion Loss provided by CompressAI
criterion = RateDistortionLoss(lmbda=LAMBDA_RD)

print(model)

In [ ]:
# Training Loop
def train_epoch(epoch, dataloader, model, criterion, optimizer):
    model.train()
    epoch_loss = 0.0
    for i, (images, _) in enumerate(dataloader):
        images = images.to(DEVICE)
        optimizer.zero_grad()
        
        out_net = model(images)
        out_criterion = criterion(out_net, images)
        
        out_criterion["loss"].backward()
        
        # Gradient clipping for stability in entropy bottlenecks
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += out_criterion["loss"].item()
        
        if i % 10 == 0:
            print(f"Epoch {epoch} | Batch {i} | Loss: {out_criterion['loss'].item():.4f} | BppLoss: {out_criterion['bpp_loss'].item():.4f} | MSELoss: {out_criterion['mse_loss'].item():.4f}")

    print(f"====> Epoch {epoch} Average Loss: {epoch_loss / max(len(dataloader), 1):.4f}")

# Uncomment to train:
# for epoch in range(1, EPOCHS + 1):
#     train_epoch(epoch, train_loader, model, criterion, optimizer)

## Ablation Study Configuration
To validate the components, you can modify the `model.py` and run tests:
1. **Baseline**: Turn off CBAM and Spatial Gate.
2. **w/o Overlap**: Change `stride=patch_size` in tokenization to show checkerboard artifacts.
3. **Adaptive Quantizer Off**: Remove `y_scaled` and entropy code `y` directly.